In [5]:
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import numpy as np
import pandas as pd

In [2]:
import numpy as np
def MG_generate(gamma=1,beta=2,tau=60,theta=1,n=10,x0=1,N=2000000,delta=0.001, interp = True):
  def MG_eq (x,x_pre):
    return x_pre * beta * (theta**n)/(theta**n+x_pre**n)-gamma*x

  def MG_rk4_interp(x,x_pre,x_pre_f):
    interplot = (x_pre+x_pre_f)/2
    k1 = MG_eq(x,x_pre)
    k2 = MG_eq(x+delta*k1/2,interplot)
    k3 = MG_eq(x+delta*k2/2,interplot)
    k4 = MG_eq(x+delta*k3,x_pre_f)
    return x+delta*(k1+2*k2+2*k3+k4)/6
  
  def MG_rk4(x,x_pre):
    k1 = MG_eq(x,x_pre)
    k2 = MG_eq(x+delta*k1/2,x_pre)
    k3 = MG_eq(x+delta*k2/2,x_pre)
    k4 = MG_eq(x+delta*k3,x_pre)
    return x+delta*(k1+2*k2+2*k3+k4)/6

  past_len = int(np.floor(tau/delta))
  x_past = np.zeros(past_len+N+1)+1.2
  x = x0
  X = np.zeros(N+1)
  T = np.zeros(N+1)
  time = 0

  for i in range(N+1):
    X[i] = x
    x_pre = x_past[i]
    x_pre_f = x_past[i+1]
    if interp:
      x_delta = MG_rk4_interp(x=x,x_pre=x_pre,x_pre_f = x_pre_f)
    else:
      x_delta = MG_rk4(x=x,x_pre=x_pre)
    x_past[i+past_len] = x_delta
    T[i] = time
    time += delta
    x = x_delta

  interval = int(np.floor(1/delta))
  T,X = T[::interval],X[::interval]
  return T,X

In [4]:
_,X = MG_generate(tau = 2, n = 9, N = 1500000,delta = 0.01,x0 = 1)

In [8]:
data = {'9':np.array(X)}
df = pd.DataFrame(data)
df.to_parquet('data_vary_n.parquet')